# Fine-tune the v11 network with pawn structure inputs (Google Colab)

This notebook trains the next network for the engine by **continuing from v11** instead of starting from zero:

- **Same size as now:** 768 piece inputs -> 1024 hidden neurons per side -> 1 output, clipped ReLU. The fast int16 code of the engine works with it.
- **New: 384 pawn structure inputs.** For every square: is there a *passed*, *isolated* or *doubled* pawn, for the own side and for the enemy (3 kinds x 2 sides x 64 squares). The engine computes exactly the same thing (`pawn_structure` in `nnue.rs`). The new weights start at **0**, so at the start this network gives exactly the v11 evaluation and only learns what the pawn inputs add.
- **Data (about 190 million positions):**
  - about 143 million quiet positions from the Lichess evaluation database (Stockfish depth >= 25), target = the score
  - 50 million positions from Lichess games (`lichess_games.bin` from notebook 07), target = 70% score + 30% **game result**

## Before you start

In **Google Drive** make a folder `bigbotturtle` in *My Drive* and put these two files in it:

- `lichess_games.bin` (the output of notebook 07)
- `nnue_weights.npz` (the v11 weights, from the `models` folder of the project)

If the files are somewhere else in Drive, the notebook searches all of Drive for them (also inside a zip file) and prints which files it uses.

Drive needs about **7 GB free**: the old Lichess positions are saved there once as a cache (about 5 GB), so they never have to be downloaded again.

## Step 1: CPU runtime (does not use GPU time)

*Runtime > Change runtime type > CPU*, then *Runtime > Run all* and allow access to Drive. The notebook runs a quick check, then streams the Lichess evaluation database and saves the cache in Drive. This takes a few hours, only once. It ends with the message that the cache is ready.

## Step 2: GPU runtime

*Runtime > Change runtime type > T4 GPU*, then *Run all* again. The notebook loads the cache and the games, trains `num_epochs` epochs and saves a checkpoint in Drive **after every epoch**. If Colab stops the session: just *Run all* again, it continues after the last finished epoch.

## Output (in `bigbotturtle/output` in Drive)

- `nnue.bin`: the engine file (format NNU2 with pawn inputs), ready for the Rust engine
- `nnue_weights.npz`: the weights, to continue training later
- `nnue_finetune.log`, `nnue_finetune.json` and `nnue_finetune_curves.png`

Keep the browser tab open while it runs: Colab stops sessions where nobody is active for a long time.

In [ ]:
!pip install -q zstandard

In [ ]:
import collections
import json
import math
import multiprocessing
import os
import shutil
import struct
import time
import urllib.request
import zipfile

import numpy as np
import torch
import torch.nn.functional as F
import zstandard
from google.colab import drive
from matplotlib import pyplot as plt
from torch import nn

drive.mount("/content/drive")

## Settings

In [ ]:
notebook_start = time.time()

drive_dir = "/content/drive/MyDrive/bigbotturtle"
drive_roots = ["/content/drive/MyDrive", "/content/drive/Othercomputers", "/content/drive/Shareddrives"]
cache_dir = os.path.join(drive_dir, "lichess_evals_cache")
out_dir = os.path.join(drive_dir, "output")

max_lichess_positions = 150_000_000
max_load_minutes = 600
min_depth = 25
wdl_weight = 0.3
use_pawn_features = True

val_size = 2_000_000
test_size = 2_000_000
batch_size = 4096
gpu_data_fraction = 0.6

learning_rate = 0.0003
num_epochs = 6
session_hours = 11.5
reserve_minutes = 20

CP_SCALE = 0.00184104
PIECE_FEATURES = 768
PAWN_FEATURES = 384
NUM_FEATURES = PIECE_FEATURES + PAWN_FEATURES if use_pawn_features else PIECE_FEATURES

url = "https://database.lichess.org/lichess_db_eval.jsonl.zst"
log_file = os.path.join(out_dir, "nnue_finetune.log")
checkpoint_file = os.path.join(out_dir, "checkpoint.pt")
best_file = os.path.join(out_dir, "best_model.pt")
engine_path = os.path.join(out_dir, "nnue.bin")
weights_path = os.path.join(out_dir, "nnue_weights.npz")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
num_workers = os.cpu_count()

print("device:", device, "| CPU cores:", num_workers)

## Helpers

In [ ]:
os.makedirs(out_dir, exist_ok=True)


def log(text):
    hours = (time.time() - notebook_start) / 3600
    line = f"[{time.strftime('%H:%M:%S')} | {hours:.2f} h] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize()


def find_file(name):
    wanted = os.path.join(drive_dir, name)
    if os.path.exists(wanted):
        return wanted
    archives = []
    for root in filter(os.path.isdir, drive_roots):
        for folder, _, files in os.walk(root):
            if folder.startswith(out_dir) or folder.startswith(cache_dir):
                continue
            if name in files:
                found = os.path.join(folder, name)
                if os.path.getsize(found) < 100_000_000:
                    shutil.copy(found, wanted)
                    return wanted
                return found
            archives += [os.path.join(folder, file) for file in files if file.lower().endswith(".zip")]
    for archive_path in archives:
        with zipfile.ZipFile(archive_path) as archive:
            member = next((member for member in archive.namelist() if os.path.basename(member) == name), None)
            if member is not None:
                log(f"Extracting {name} from {archive_path} to {drive_dir} (only once)...")
                with archive.open(member) as source, open(wanted, "wb") as target:
                    shutil.copyfileobj(source, target, 16 << 20)
                return wanted
    return None


log(f"New session on {device}, looking for the input files in Google Drive...")
games_file = find_file("lichess_games.bin")
start_weights_file = find_file("nnue_weights.npz")
for name, path in (("lichess_games.bin", games_file), ("nnue_weights.npz", start_weights_file)):
    top_level = sorted(os.listdir("/content/drive/MyDrive"))
    assert path is not None, f"{name} not found anywhere in Google Drive (also not in a zip file). Is the upload finished? Top level of My Drive: {top_level}"
log(f"Using {games_file} ({os.path.getsize(games_file) / 1e9:.2f} GB) and {start_weights_file}")

## Old data: quiet positions with a deep evaluation, streamed from Lichess once and kept as a cache in Drive

The same filter as notebooks 05 and 06: Stockfish depth >= 25, and the best move is not a capture or promotion. The label is the score as a number between -1 and 1 (`tanh` of the centipawns).

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"
CACHE_NAMES = ["packed", "side", "labels"]


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    index = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            index += int(char)
        else:
            row[index ^ 56] = PIECE_CODES[char]
            index += 1
    return row, color == "w"


def square_index(square):
    return (int(square[1]) - 1) * 8 + (ord(square[0]) - ord("a"))


def is_quiet(row, move):
    if len(move) == 5:
        return False
    from_square, to_square = square_index(move[:2]), square_index(move[2:4])
    if row[to_square] != 0:
        return False
    if row[from_square] in (1, 7) and move[0] != move[2]:
        return False
    return True


def score_to_label(cp, mate):
    if mate is not None:
        return 1.0 if mate > 0 else -1.0
    return 2 / (1 + math.exp(-0.00368208 * cp)) - 1


def parse_chunk(chunk):
    rows, sides, values = [], [], []
    num_lines = 0
    for line in chunk.splitlines():
        num_lines += 1
        data = json.loads(line)
        if not data["evals"]:
            continue
        deepest = max(data["evals"], key=lambda e: e["depth"])
        pv = deepest["pvs"][0]
        if deepest["depth"] < min_depth or not pv.get("line"):
            continue
        row, white = parse_fen(data["fen"])
        if not is_quiet(row, pv["line"].split()[0]):
            continue
        rows.append(row[0::2] | (row[1::2] << 4))
        sides.append(white)
        values.append(score_to_label(pv.get("cp"), pv.get("mate")))
    packed = np.array(rows, dtype=np.uint8).reshape(-1, 32)
    return packed, np.array(sides, dtype=np.uint8), np.array(values, dtype=np.float16), num_lines


def read_chunks(reader, chunk_size=16_000_000):
    rest = b""
    while True:
        block = reader.read(chunk_size)
        if not block:
            break
        block = rest + block
        end = block.rfind(b"\n") + 1
        rest = block[end:]
        yield block[:end]
    if rest:
        yield rest


def load_positions(max_positions, max_minutes):
    packed = np.zeros((max_positions, 32), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float16)
    count, num_lines, next_log = 0, 0, 5_000_000
    start = time.time()

    with urllib.request.urlopen(url) as response, multiprocessing.get_context("fork").Pool(num_workers) as pool:
        chunks = read_chunks(zstandard.ZstdDecompressor().stream_reader(response))
        pending = collections.deque()
        while count < max_positions and time.time() - start < max_minutes * 60:
            while len(pending) < 2 * num_workers:
                chunk = next(chunks, None)
                if chunk is None:
                    break
                pending.append(pool.apply_async(parse_chunk, (chunk,)))
            if not pending:
                break

            chunk_packed, chunk_side, chunk_labels, chunk_lines = pending.popleft().get()
            size = min(len(chunk_labels), max_positions - count)
            packed[count:count + size] = chunk_packed[:size]
            side[count:count + size] = chunk_side[:size]
            labels[count:count + size] = chunk_labels[:size]
            count += size
            num_lines += chunk_lines
            if count >= next_log:
                log(f"Loaded {count:,} / {max_positions:,} positions from {num_lines:,} lines")
                next_log += 5_000_000

    log(f"Done: {count:,} positions from {num_lines:,} lines in {(time.time() - start) / 60:.0f} min")
    return packed[:count], side[:count], labels[:count]


def cache_ready():
    return os.path.exists(os.path.join(cache_dir, "done.json"))


def save_cache(arrays):
    os.makedirs(cache_dir, exist_ok=True)
    for name, array in zip(CACHE_NAMES, arrays):
        np.save(os.path.join(cache_dir, f"{name}.npy"), array)
    with open(os.path.join(cache_dir, "done.json"), "w", encoding="utf-8") as f:
        json.dump({"positions": len(arrays[2]), "min_depth": min_depth}, f)


def open_cache():
    return tuple(np.load(os.path.join(cache_dir, f"{name}.npy"), mmap_mode="r") for name in CACHE_NAMES)

## New data: positions from Lichess games (notebook 07)

Each record is 36 bytes: the board (32 bytes, the same packing as above), the side to move, the Stockfish score in centipawns for white (mate = ±10000) and the result of the game (0 black won, 1 draw, 2 white won). The target is `(1 - wdl_weight) * score + wdl_weight * result`.

In [ ]:
GAME_RECORD = np.dtype([("board", "u1", 32), ("white", "u1"), ("score", "<i2"), ("result", "u1")])


def open_games():
    return np.memmap(games_file, dtype=GAME_RECORD, mode="r")


def game_labels(records):
    score_labels = 2 / (1 + np.exp(-0.00368208 * records["score"].astype(np.float32))) - 1
    result_labels = records["result"].astype(np.float32) - 1
    return ((1 - wdl_weight) * score_labels + wdl_weight * result_labels).astype(np.float16)

## Features: 768 piece inputs + 384 pawn structure inputs per perspective

Each perspective sees the board from its own side: for black the board is flipped, so "own" and "enemy" always mean the same thing.

The pawn structure is computed on the GPU for a whole batch at once, on an 8x8 grid per board (rank, file):

- **passed:** no enemy pawn in front of it on the same or an adjacent file
- **isolated:** no own pawn on an adjacent file
- **doubled:** another own pawn on the same file

The input number is `768 + kind * 128 + (0 own, 64 enemy) + square`, exactly like `pawn_feature` in the engine.

In [ ]:
PIECE_TYPES = torch.tensor([0, 0, 1, 2, 3, 4, 5, 0, 1, 2, 3, 4, 5], device=device)
PIECE_COLORS = torch.tensor([0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], device=device)
SQUARES = torch.arange(64, device=device)
PAWN_KINDS = torch.arange(3, device=device).view(1, 3, 1) * 128


def unpack(data_packed):
    low = data_packed & 15
    high = data_packed >> 4
    return torch.stack([low, high], dim=2).reshape(-1, 64).long()


def north(x):
    return (F.pad(x[:, :-1, :], (0, 0, 1, 0)).cumsum(dim=1) > 0).int()


def south(x):
    return north(x.flip(1)).flip(1)


def adjacent(x):
    return F.pad(x[:, :, :-1], (1, 0)) | F.pad(x[:, :, 1:], (0, 1))


def pawn_structure(board):
    pawns = [(board == 1).int().view(-1, 8, 8), (board == 7).int().view(-1, 8, 8)]
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south(enemy) if color == 0 else north(enemy)
        files = own | north(own) | south(own)
        passed = own & (1 - (blockers | adjacent(blockers)))
        isolated = own & (1 - adjacent(files))
        doubled = own & (north(own) | south(own))
        structure.append(torch.stack([passed, isolated, doubled], dim=1).view(-1, 3, 64))
    return structure


def perspective_features(board, structure, perspective):
    flipped = SQUARES ^ (56 * perspective)
    own = (PIECE_COLORS[board] != perspective).long() * 384
    pieces = (own + PIECE_TYPES[board] * 64 + flipped).masked_fill(board == 0, NUM_FEATURES)
    if not use_pawn_features:
        return pieces.sort(dim=1).values[:, :32]
    parts = [pieces]
    for color in (0, 1):
        index = PIECE_FEATURES + (0 if color == perspective else 64) + PAWN_KINDS + flipped.view(1, 1, 64)
        parts.append(torch.where(structure[color].bool(), index, NUM_FEATURES).view(-1, 192))
    x = torch.cat(parts, dim=1).sort(dim=1).values
    return x[:, :int((x < NUM_FEATURES).sum(dim=1).max())]


def features(board, white):
    structure = pawn_structure(board) if use_pawn_features else None
    x_white = perspective_features(board, structure, 0)
    x_black = perspective_features(board, structure, 1)
    width = max(x_white.shape[1], x_black.shape[1])
    x_white = F.pad(x_white, (0, width - x_white.shape[1]), value=NUM_FEATURES)
    x_black = F.pad(x_black, (0, width - x_black.shape[1]), value=NUM_FEATURES)
    white = white.bool().unsqueeze(1)
    return torch.where(white, x_white, x_black), torch.where(white, x_black, x_white)

## Model: v11 plus pawn inputs

The same shape as v11: the inputs are added up per perspective (`EmbeddingBag`, one row per input), clipped between 0 and 1, and one linear layer gives the output. During training the output goes through `tanh`, just like the labels; the engine uses the value before `tanh`.

`build_model` copies the v11 weights into the model. The rows of the pawn inputs stay 0.

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_hidden):
        super().__init__()
        self.feature_transformer = nn.EmbeddingBag(NUM_FEATURES + 1, num_hidden, mode="sum", padding_idx=NUM_FEATURES)
        self.feature_bias = nn.Parameter(torch.zeros(num_hidden))
        self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us) + self.feature_bias, 0, 1)
        them = torch.clamp(self.feature_transformer(x_them) + self.feature_bias, 0, 1)
        x = torch.cat([us, them], dim=1)
        return self.output(x)[:, 0]


def build_model(start):
    weight = start["feature_transformer.weight"]
    assert weight.shape[1] <= NUM_FEATURES, "the start weights have more inputs than this model"
    model = NNUE(num_hidden=weight.shape[0])
    with torch.no_grad():
        model.feature_transformer.weight.zero_()
        model.feature_transformer.weight[:weight.shape[1]] = torch.from_numpy(weight.T.copy())
        model.feature_bias.copy_(torch.from_numpy(start["feature_transformer.bias"]))
        model.output.weight.copy_(torch.from_numpy(start["output.weight"]))
        model.output.bias.copy_(torch.from_numpy(start["output.bias"]))
    return model.to(device)

## Engine file (nnue.bin, format NNU2 with pawn inputs) and a Python version of the engine evaluation

The pawn structure here is a line-by-line copy of the Rust code (bitboards with `north_span`, `south_span` and `adjacent_files`). Before this notebook was made, this Python evaluation was compared with the Rust engine on 5,000 positions from engine matches (4,550 passed, 7,487 isolated and 2,478 doubled pawns): the largest difference was 0.00000.

In [ ]:
FILE_A = 0x0101010101010101
FILE_H = FILE_A << 7
FULL = (1 << 64) - 1


def write_nnue(net, path):
    num_features, num_hidden = net["feature_weights"].shape
    with open(path, "wb") as f:
        f.write(b"NNU2")
        f.write(struct.pack("<5I", 1, 0, int(num_features > PIECE_FEATURES), num_hidden, 0))
        f.write(struct.pack("<64I", *([0] * 64)))
        for part in (net["feature_weights"], net["feature_bias"], np.zeros(num_features), net["output_weight"], net["output_bias"]):
            f.write(np.asarray(part, dtype="<f4").tobytes())


def read_nnue(path):
    with open(path, "rb") as f:
        data = f.read()
    assert data[:4] == b"NNU2", "not an NNU2 file"
    num_buckets, mirror, pawns, num_hidden, num_layer2 = struct.unpack("<5I", data[4:24])
    assert (num_buckets, mirror, num_layer2) == (1, 0, 0), "unexpected NNU2 header"
    num_features = PIECE_FEATURES + PAWN_FEATURES * pawns
    values = np.frombuffer(data, dtype="<f4", offset=24 + 64 * 4)
    sizes = [num_features * num_hidden, num_hidden, num_features, 2 * num_hidden, 1]
    assert sum(sizes) == len(values), "unexpected file size"
    parts = np.split(values, np.cumsum(sizes)[:-1])
    return {
        "feature_weights": parts[0].reshape(num_features, num_hidden),
        "feature_bias": parts[1],
        "output_weight": parts[3].reshape(1, 2 * num_hidden),
        "output_bias": parts[4],
    }


def north_span(squares):
    span = (squares << 8) & FULL
    span |= (span << 8) & FULL
    span |= (span << 16) & FULL
    return span | ((span << 32) & FULL)


def south_span(squares):
    span = squares >> 8
    span |= span >> 8
    span |= span >> 16
    return span | (span >> 32)


def adjacent_files(squares):
    return ((squares << 1) & ~FILE_A & FULL) | ((squares >> 1) & ~FILE_H)


def reference_pawn_structure(row):
    pawns = [0, 0]
    for square in range(64):
        if row[square] == 1:
            pawns[0] |= 1 << square
        elif row[square] == 7:
            pawns[1] |= 1 << square
    structure = []
    for color in (0, 1):
        own, enemy = pawns[color], pawns[1 - color]
        blockers = south_span(enemy) if color == 0 else north_span(enemy)
        files = own | north_span(own) | south_span(own)
        passed = own & ~(blockers | adjacent_files(blockers))
        isolated = own & ~adjacent_files(files)
        doubled = own & (north_span(own) | south_span(own))
        structure.append([passed, isolated, doubled])
    return structure


def reference_features(row, perspective):
    flip = 56 * perspective
    active = []
    for square in range(64):
        code = int(row[square])
        if code == 0:
            continue
        own = 0 if (code <= 6) == (perspective == 0) else 384
        active.append(own + (code - 1) % 6 * 64 + (square ^ flip))
    if use_pawn_features:
        for color, kinds in enumerate(reference_pawn_structure(row)):
            own = 0 if color == perspective else 64
            for kind, squares in enumerate(kinds):
                for square in range(64):
                    if squares >> square & 1:
                        active.append(PIECE_FEATURES + kind * 128 + own + (square ^ flip))
    return active


def reference_eval(net, row, white):
    us, them = (0, 1) if white else (1, 0)
    hidden = []
    for perspective in (0, 1):
        active = reference_features(row, perspective)
        hidden.append(np.clip(net["feature_bias"] + net["feature_weights"][active].sum(axis=0), 0, 1))
    x = np.concatenate([hidden[us], hidden[them]])
    return float((net["output_weight"] @ x + net["output_bias"])[0])

In [ ]:
def model_net(model):
    weights = {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}
    return {
        "feature_weights": weights["feature_transformer.weight"][:NUM_FEATURES],
        "feature_bias": weights["feature_bias"],
        "output_weight": weights["output.weight"],
        "output_bias": weights["output.bias"],
    }


def start_net(start):
    weight = start["feature_transformer.weight"].T
    feature_weights = np.zeros((NUM_FEATURES, weight.shape[1]), dtype=np.float32)
    feature_weights[:len(weight)] = weight
    return {
        "feature_weights": feature_weights,
        "feature_bias": start["feature_transformer.bias"],
        "output_weight": start["output.weight"],
        "output_bias": start["output.bias"],
    }


def save_weights(model, path):
    net = model_net(model)
    np.savez(
        path,
        num_hidden=np.array(net["feature_bias"].shape[0]),
        num_layer2=np.array(0),
        **{
            "feature_transformer.weight": net["feature_weights"].T,
            "feature_transformer.bias": net["feature_bias"],
            "output.weight": net["output_weight"],
            "output.bias": net["output_bias"],
        },
    )


def export_model(model, path):
    write_nnue(model_net(model), path)


def predict(model, row, white):
    board = torch.from_numpy(np.asarray(row)).long().unsqueeze(0).to(device)
    data_white = torch.tensor([white], device=device)
    model.eval()
    with torch.no_grad():
        x_us, x_them = features(board, data_white)
        return model(x_us, x_them).item()

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def batches(data, indices, shuffle):
    data_packed, data_side, data_labels = data
    if shuffle:
        indices = indices[torch.randperm(len(indices), device=indices.device)]
    for start in range(0, len(indices), batch_size):
        batch = indices[start:start + batch_size].long()
        board = unpack(data_packed[batch].to(device, non_blocking=True))
        white = data_side[batch].to(device, non_blocking=True).bool()
        x_us, x_them = features(board, white)
        labels = data_labels[batch].to(device, non_blocking=True).float()
        targets = torch.where(white, labels, -labels)
        yield x_us, x_them, targets


def train_one_epoch(model, optimizer, data, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, targets in batches(data, indices, shuffle=True):
        preds = torch.tanh(model(x_us, x_them))
        loss = loss_fn(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.detach()
        num_batches += 1

    return (total_loss / num_batches).item()


def eval_model(model, data, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, targets in batches(data, indices, shuffle=False):
            preds = torch.tanh(model(x_us, x_them))

            total_loss += loss_fn(preds, targets)
            num_batches += 1

            total_abs_error += (preds - targets).abs().sum()
            num_preds += targets.shape[0]

    return (total_loss / num_batches).item(), (total_abs_error / num_preds).item()


def measure_speed(learning_rate, data, indices, num_batches=300, warmup=50):
    model = build_model(start_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    model.train()

    for i, (x_us, x_them, targets) in enumerate(batches(data, indices, shuffle=True)):
        if i == warmup:
            synchronize()
            start = time.time()
        if i == num_batches:
            break
        loss = loss_fn(torch.tanh(model(x_us, x_them)), targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    synchronize()
    return (num_batches - warmup) * batch_size / (time.time() - start)

## Quick check of the whole pipeline

Fails within minutes if something is wrong:

- the model starts with exactly the v11 evaluation
- the pawn and piece inputs on the GPU are the same as in the Python copy of the engine
- training works, and the exported `nnue.bin` gives the same evaluation as the model
- the games file has exactly one king per side in every position

In [ ]:
log("Quick check of the whole pipeline on a small sample...")
start_weights = dict(np.load(start_weights_file))
games = open_games()
log(f"lichess_games.bin: {len(games):,} positions, v11 start weights: {start_weights['feature_transformer.weight'].shape[1]} inputs -> {start_weights['feature_transformer.weight'].shape[0]} hidden")

sample_games = np.array(games[:20_000])
sample = (
    torch.from_numpy(np.ascontiguousarray(sample_games["board"])).to(device),
    torch.from_numpy(sample_games["white"].copy()).to(device),
    torch.from_numpy(game_labels(sample_games)).to(device),
)
sample_idx = torch.arange(len(sample[2]), device=device, dtype=torch.int32)
board = unpack(sample[0][:1000])
assert bool(((board == 6).sum(dim=1) == 1).all()) and bool(((board == 12).sum(dim=1) == 1).all()), "every position needs exactly one king per side"

check_model = build_model(start_weights)
v11_net = start_net(start_weights)
feature_errors, start_error = 0, 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    x_us, x_them = features(torch.from_numpy(row).long().unsqueeze(0).to(device), torch.tensor([white], device=device))
    us = 0 if white else 1
    for x, perspective in [(x_us, us), (x_them, 1 - us)]:
        if sorted(index for index in x[0].tolist() if index < NUM_FEATURES) != sorted(reference_features(row, perspective)):
            feature_errors += 1
    start_error = max(start_error, abs(predict(check_model, row, white) - reference_eval(v11_net, row, white)))
log(f"Feature mismatches: {feature_errors}, largest difference between the start model and v11: {start_error:.6f}")
assert feature_errors == 0, "the features differ from the engine"
assert start_error < 1e-4, "the start model is not v11"

check_optimizer = torch.optim.Adam(check_model.parameters(), lr=1e-3)
train_loss = train_one_epoch(check_model, check_optimizer, sample, sample_idx)
val_loss, val_mae = eval_model(check_model, sample, sample_idx)
log(f"One epoch on the sample: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}")
log(f"Speed: {measure_speed(1e-3, sample, sample_idx, num_batches=4, warmup=1):,.0f} positions/s")

with torch.no_grad():
    for param in check_model.parameters():
        param.add_(0.01 * torch.randn_like(param))
check_path = os.path.join(out_dir, "check.bin")
export_model(check_model, check_path)
check_net = read_nnue(check_path)
os.remove(check_path)
export_error = 0.
for i in range(300):
    row = unpack(sample[0][i:i + 1])[0].cpu().numpy()
    white = bool(sample[1][i])
    export_error = max(export_error, abs(predict(check_model, row, white) - reference_eval(check_net, row, white)))
log(f"Largest difference between the model and the engine file: {export_error:.6f}")
assert export_error < 1e-3, "the engine file does not match the model"

if not cache_ready():
    lichess_sample = load_positions(20_000, max_minutes=10)
    assert len(lichess_sample[2]) > 1000, "could not read the Lichess evaluation database"
log("Quick check passed")

## Data

The first time (best on a CPU runtime) the Lichess evaluation database is streamed and saved as a cache in Drive. After that the cache and the games are put together in one dataset. Everything is kept on the GPU if it fits, so training does not wait for the CPU.

The split into train, validation and test uses a fixed seed, so it is the same in every session.

In [ ]:
if not cache_ready():
    log(f"No cache yet: streaming the Lichess evaluation database with {num_workers} workers (only once, this takes a few hours)...")
    save_cache(load_positions(max_lichess_positions, max_load_minutes))
    log(f"Cache saved in {cache_dir}")
assert device.type == "cuda", "The data cache is ready in Google Drive. Now switch to a GPU runtime (Runtime > Change runtime type > T4 GPU) and run all cells again."

cache = open_cache()
num_lichess, num_games = len(cache[2]), len(games)
num_positions = num_lichess + num_games
arrays = (
    np.empty((num_positions, 32), dtype=np.uint8),
    np.empty(num_positions, dtype=np.uint8),
    np.empty(num_positions, dtype=np.float16),
)
chunk = 10_000_000
for start in range(0, num_lichess, chunk):
    end = min(start + chunk, num_lichess)
    for array, cached in zip(arrays, cache):
        array[start:end] = cached[start:end]
for start in range(0, num_games, chunk):
    part = games[start:start + chunk]
    first, last = num_lichess + start, num_lichess + start + len(part)
    arrays[0][first:last] = part["board"]
    arrays[1][first:last] = part["white"]
    arrays[2][first:last] = game_labels(part)
del cache, games, sample, sample_games

data_gb = sum(array.nbytes for array in arrays) / 1e9
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
data_device = device if data_gb < gpu_data_fraction * gpu_gb else torch.device("cpu")
data = tuple(torch.from_numpy(array).to(data_device) for array in arrays)
del arrays
log(f"Data: {num_positions:,} positions ({num_lichess:,} Lichess evaluations, {num_games:,} from games), {data_gb:.1f} GB, kept on {data_device}")

perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).int().to(data_device)
val_idx = perm[:val_size]
test_idx = perm[val_size:val_size + test_size]
train_idx = perm[val_size + test_size:]
del perm
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")
log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {gpu_gb:.1f} GB")

## Training (continues from the checkpoint if there is one)

Adam with a cosine learning rate schedule over `num_epochs` epochs. The learning rate is lower than when v11 was trained from zero, because the network only has to adjust, not learn chess from the start.

After every epoch the checkpoint is saved in Drive. When the validation loss is the best so far, `nnue.bin` and `nnue_weights.npz` are saved as well.

In [ ]:
deadline = notebook_start + session_hours * 3600 - reserve_minutes * 60
speed = measure_speed(learning_rate, data, train_idx)
epoch_seconds = 1.1 * len(train_idx) / speed
log(f"Speed: {speed:,.0f} positions/s, about {epoch_seconds / 60:.0f} min per epoch, {num_epochs} epochs in total")

model = build_model(start_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=learning_rate / 100)
history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")

if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    history = checkpoint["history"]
    best_val_loss = checkpoint["best_val_loss"]
    log(f"Continuing from the checkpoint after epoch {len(history['train_loss'])}")
else:
    start_val_loss, start_val_mae = eval_model(model, data, val_idx)
    log(f"Before training (v11, pawn inputs at 0): val loss {start_val_loss:.4f}, val MAE {start_val_mae:.4f}")
print(model)

for epoch in range(len(history["train_loss"]), num_epochs):
    if time.time() + epoch_seconds > deadline:
        log("Not enough session time left for another epoch: run all cells again in a new session to continue")
        break

    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, data, train_idx)
    val_loss, val_mae = eval_model(model, data, val_idx)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}/{num_epochs}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_file)
        save_weights(model, weights_path)
        export_model(model, engine_path)
        log("Best so far: saved nnue.bin and nnue_weights.npz in Drive")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "history": history, "best_val_loss": best_val_loss}, checkpoint_file)

finished = len(history["train_loss"]) == num_epochs
log("Training finished" if finished else f"Training paused after epoch {len(history['train_loss'])} of {num_epochs}")

## Test

The best network on the test positions, compared with v11 on the same positions.

In [ ]:
assert finished, "Training is not finished yet: run all cells again in a new session to continue from the checkpoint"
model.load_state_dict(torch.load(best_file, map_location=device))
test_loss, test_mae = eval_model(model, data, test_idx)
v11_test_loss, v11_test_mae = eval_model(build_model(start_weights), data, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f} (v11: loss {v11_test_loss:.4f}, MAE {v11_test_mae:.4f})")

## Sanity check, piece values and what the pawn inputs learned

The piece values are measured like this: the evaluation of a position where black misses one piece, minus the evaluation of the same position with all pieces. For comparison, v11 gave: opening pawn +64, knight +363, bishop +417, rook +446, queen +667; endgame pawn +165, 2 pawns +368, rook +923.

The pawn part is the evaluation of the network minus the evaluation of the same network with the pawn input weights set to 0.

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "rook against 3 pawns, white to move (about +200)": "1nbqkbnr/pppppppp/8/8/8/8/3PPPPP/RNBQKBNR w KQk - 0 1",
    "lost game 1, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
    "lost game 2, black to move (Stockfish -387)": "2krr3/1pp2p1p/p1nb3q/3p4/Q1PP2pN/3bP1P1/P2N1PBP/R2R2K1 b - - 0 16",
}

engine_net = read_nnue(engine_path)
sanity = {}
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    raw = predict(model, row, white)
    engine_raw = reference_eval(engine_net, row, white)
    sanity[name] = {"fen": fen, "raw": raw}
    log(f"{name}: {raw / CP_SCALE:+.0f} cp for the side to move (engine file: {engine_raw / CP_SCALE:+.0f} cp)")


material_tests = {
    "opening": ("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1", {
        "pawn": "rnbqkbnr/1ppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "knight": "r1bqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "bishop": "rn1qkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
        "rook": "1nbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQk - 0 1",
        "queen": "rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    }),
    "endgame": ("r3k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1", {
        "pawn": "r3k3/ppp2pp1/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "2 pawns": "r3k3/ppp2p2/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
        "rook": "4k3/ppp2ppp/8/8/8/8/PPP2PPP/R3K3 w - - 0 1",
    }),
}

piece_values = {}
for phase, (start_fen, missing) in material_tests.items():
    start_value = predict(model, *parse_fen(start_fen))
    for name, fen in missing.items():
        value = (predict(model, *parse_fen(fen)) - start_value) / CP_SCALE
        piece_values[f"{phase}: black without {name}"] = value
        log(f"{phase}, black without {name}: {value:+.0f} cp for white")

pawn_tests = {
    "white passed pawn on d6, white to move": "6k1/5ppp/3P4/8/8/8/5PPP/6K1 w - - 0 1",
    "black isolated pawn on d5, black to move": "r1bq1rk1/pp3ppp/2n2n2/3p4/3N4/2N1B3/PP2BPPP/R2Q1RK1 b - - 0 10",
    "white doubled c-pawns, black to move": "r1bqk2r/pppp1ppp/2n2n2/4p3/4P3/2P2N2/P1PP1PPP/R1BQKB1R b KQkq - 0 5",
}
pawn_effects = {}
if use_pawn_features:
    no_pawn_model = build_model(start_weights)
    no_pawn_model.load_state_dict(model.state_dict())
    with torch.no_grad():
        no_pawn_model.feature_transformer.weight[PIECE_FEATURES:NUM_FEATURES] = 0
    for name, fen in pawn_tests.items():
        row, white = parse_fen(fen)
        total, without = predict(model, row, white), predict(no_pawn_model, row, white)
        pawn_effects[name] = (total - without) / CP_SCALE
        log(f"{name}: {total / CP_SCALE:+.0f} cp for the side to move, of which the pawn inputs {pawn_effects[name]:+.0f} cp")

params = {
    "num_hidden": model.feature_bias.shape[0],
    "num_features": NUM_FEATURES,
    "use_pawn_features": use_pawn_features,
    "activation": "clipped relu",
    "start": "v11",
    "learning_rate": learning_rate,
    "wdl_weight": wdl_weight,
    "num_lichess": num_lichess,
    "num_games": num_games,
    "min_depth": min_depth,
    "num_epochs": len(history["train_loss"]),
    "positions_per_second": speed,
}
with open(os.path.join(out_dir, "nnue_finetune.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae, "v11_test_loss": v11_test_loss, "v11_test_mae": v11_test_mae, "sanity": sanity, "piece_values": piece_values, "pawn_effects": pawn_effects}, f, indent=2)
log("Saved nnue_finetune.json")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_finetune_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
log(f"Done. Download nnue.bin from {out_dir} for the engine. Files: {sorted(os.listdir(out_dir))}")